# 02 — Markov Chains
**Week 2 | Mathematical Foundations for RL**

Markov chains are the mathematical backbone of MDPs. The **Markov property** states:

$$P(s_{t+1} | s_t, s_{t-1}, ..., s_0) = P(s_{t+1} | s_t)$$

The future depends only on the present — not the history. This is what makes RL tractable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
np.random.seed(0)

## 1. A Simple 3-State Markov Chain
States: Sunny (0), Cloudy (1), Rainy (2)

In [ ]:
# Transition matrix T[i,j] = P(next=j | current=i)
T = np.array([
    [0.7, 0.2, 0.1],  # from Sunny
    [0.3, 0.4, 0.3],  # from Cloudy
    [0.2, 0.3, 0.5],  # from Rainy
])

state_names = ['Sunny', 'Cloudy', 'Rainy']

# Verify rows sum to 1
assert np.allclose(T.sum(axis=1), 1), "Rows must sum to 1!"
print("Transition matrix:")
print(T)

## 2. Simulate a Trajectory

In [ ]:
def simulate_markov(T, start_state, n_steps):
    trajectory = [start_state]
    state = start_state
    for _ in range(n_steps - 1):
        state = np.random.choice(len(T), p=T[state])
        trajectory.append(state)
    return np.array(trajectory)

traj = simulate_markov(T, start_state=0, n_steps=50)

plt.figure(figsize=(10, 2.5))
colors = ['gold', 'skyblue', 'steelblue']
plt.step(range(len(traj)), traj, where='post', color='navy', linewidth=1.5)
plt.yticks([0,1,2], state_names)
plt.xlabel('Time step'); plt.title('Weather Markov Chain — 50 steps')
plt.tight_layout(); plt.show()

## 3. Stationary Distribution
After many steps, the chain settles into a **stationary distribution** π where π = π · T.

We can find it as the left eigenvector of T corresponding to eigenvalue 1.

In [ ]:
# Analytical: left eigenvector
eigenvalues, eigenvectors = np.linalg.eig(T.T)
idx = np.argmin(np.abs(eigenvalues - 1.0))  # eigenvalue closest to 1
stationary_analytical = np.real(eigenvectors[:, idx])
stationary_analytical /= stationary_analytical.sum()
print("Analytical stationary distribution:", np.round(stationary_analytical, 4))

# Empirical: simulate 100,000 steps
long_traj = simulate_markov(T, start_state=0, n_steps=100_000)
stationary_empirical = np.bincount(long_traj) / len(long_traj)
print("Empirical  stationary distribution:", np.round(stationary_empirical, 4))

In [ ]:
# Visualise convergence to stationary distribution
n_steps = 2000
state_freq = np.zeros((n_steps, 3))
traj_long = simulate_markov(T, start_state=0, n_steps=n_steps)
for t in range(1, n_steps):
    state_freq[t] = np.bincount(traj_long[:t+1], minlength=3) / (t+1)

fig, ax = plt.subplots(figsize=(9, 3.5))
colors_ = ['gold', 'skyblue', 'steelblue']
for i, (name, c) in enumerate(zip(state_names, colors_)):
    ax.plot(state_freq[:, i], color=c, linewidth=1.5, label=name)
    ax.axhline(stationary_analytical[i], color=c, linestyle='--', alpha=0.6)
ax.set_xlabel('Steps'); ax.set_ylabel('Frequency')
ax.set_title('Convergence to Stationary Distribution (dashed = theoretical)')
ax.legend(); plt.tight_layout(); plt.show()

## 4. Matrix Power — Another View
P(state at t=n | start state) = T^n · initial_distribution

In [ ]:
init = np.array([1.0, 0.0, 0.0])  # start in Sunny
print(f"t=0:  {init}")
Tn = T.copy()
for t in [1, 5, 10, 50]:
    Tn_power = np.linalg.matrix_power(T, t)
    dist = init @ Tn_power
    print(f"t={t:<3}: {np.round(dist, 4)}")  # should converge to stationary

## ✅ Exercises
1. Change the transition matrix so that once it rains, it always rains next (absorbing state). What happens to the stationary distribution?
2. Add a 4th state (Stormy) to the chain. Update T, make sure rows sum to 1, and re-run.
3. **Challenge**: prove to yourself that `π @ T == π` for the stationary distribution you computed above. Write the assertion.

## Q1
An "absorbing state" is like a black hole — once you enter it, you can never leave. If "once it rains, it always rains," then no matter where you start, you'll eventually fall into Rainy and get stuck there forever. So the stationary distribution becomes 100% Rainy, 0% everything else — because in the long run, that's the only place the chain can be.

In [ ]:
# Modify T: Rainy becomes absorbing (row 2 -> always stays Rainy)
T_absorbing = np.array([
    [0.7, 0.2, 0.1],  # from Sunny (unchanged)
    [0.3, 0.4, 0.3],  # from Cloudy (unchanged)
    [0.0, 0.0, 1.0],  # from Rainy -> always Rainy
])

assert np.allclose(T_absorbing.sum(axis=1), 1)

# Simulate long trajectory
long_traj_abs = simulate_markov(T_absorbing, start_state=0, n_steps=10_000)
stationary_abs = np.bincount(long_traj_abs, minlength=3) / len(long_traj_abs)
print("Stationary distribution (absorbing Rainy):", np.round(stationary_abs, 4))
# Expect something close to [0, 0, 1.0]

## Q2

In [ ]:
state_names_4 = ['Sunny', 'Cloudy', 'Rainy', 'Stormy']

T4 = np.array([
    [0.6, 0.2, 0.1, 0.1],   # from Sunny
    [0.25, 0.35, 0.3, 0.1], # from Cloudy
    [0.1, 0.2, 0.5, 0.2],   # from Rainy
    [0.05, 0.15, 0.3, 0.5], # from Stormy
])

assert np.allclose(T4.sum(axis=1), 1), "Rows must sum to 1!"
print("4-state transition matrix:")
print(T4)

# Re-run simulation
traj4 = simulate_markov(T4, start_state=0, n_steps=50)

plt.figure(figsize=(10, 2.5))
plt.step(range(len(traj4)), traj4, where='post', color='navy', linewidth=1.5)
plt.yticks([0,1,2,3], state_names_4)
plt.xlabel('Time step'); plt.title('Weather Markov Chain (4 states) — 50 steps')
plt.tight_layout(); plt.show()

# Stationary distribution (analytical)
eigenvalues4, eigenvectors4 = np.linalg.eig(T4.T)
idx4 = np.argmin(np.abs(eigenvalues4 - 1.0))
stationary4 = np.real(eigenvectors4[:, idx4])
stationary4 /= stationary4.sum()
print("Stationary distribution (4 states):", np.round(stationary4, 4))

## Q3
#### Prove π @ T == π
π is a probability distribution that, once you apply one step of the transition matrix, maps back to itself. It's like a fixed point: the weather's overall probability mix doesn't change step to step, even though the individual day's weather still does.

In [ ]:
# Using the original 3-state T and stationary_analytical computed earlier
pi_next = stationary_analytical @ T

print("π        :", np.round(stationary_analytical, 6))
print("π @ T    :", np.round(pi_next, 6))

assert np.allclose(stationary_analytical @ T, stationary_analytical, atol=1e-6), \
    "Stationary distribution should be invariant under T!"
print("✅ Assertion passed: π @ T == π")

Why it should work mathematically: π is defined as the eigenvector of T.T for eigenvalue 1 — that's exactly saying T.T @ π = π, which is the same as π @ T = π (just transposed notation). So this assertion isn't a coincidence, it's confirming the eigenvector calculation was done correctly.